In [1]:
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from models_ import *
from utils import *

from flow_model import *
from flow_utils import *

from forward_inverse import *
import torch.nn.functional as F

# init

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

In [3]:
device

device(type='mps')

In [4]:
def set_seed(seed: int = 42):
    random.seed(seed)                       
    np.random.seed(seed)                   
    torch.manual_seed(seed)                

    if torch.backends.mps.is_available():
  
        print("Using MPS: Seed fixed for reproducibility")
    elif torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False   

In [5]:
set_seed(42)

Using MPS: Seed fixed for reproducibility


# save&load

In [6]:
import os
def save_complex_models(nested_dict, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    for top_key, sub_dict in nested_dict.items():
        model_folder = os.path.join(save_dir, top_key)
        os.makedirs(model_folder, exist_ok=True)

        if isinstance(sub_dict, dict):
            for name, model in sub_dict.items():
               
                if isinstance(model, dict):
                    for subname, submodel in model.items():
                        path = os.path.join(model_folder, f"{name}_{subname}.pth")
                        print(f"Saving {top_key} / {name}_{subname} to {path}")
                        torch.save(submodel.state_dict(), path)
                else:
                    path = os.path.join(model_folder, f"{name}.pth")
                    print(f"Saving {top_key} / {name} to {path}")
                    torch.save(model.state_dict(), path)
        else:
          
            path = os.path.join(save_dir, f"{top_key}.pth")
            print(f"Saving {top_key} to {path}")
            torch.save(sub_dict.state_dict(), path)

    print(f"✔️ Models saved to {save_dir}")

def load_all_model_families(model_families, save_dir="saved_vecfield_models"):

    loaded_families = {}

    for family_name, model_dict in model_families.items():
        folder_path = os.path.join(save_dir, family_name)
        if not os.path.exists(folder_path):
            print(f"❌ folder {folder_path} not found, skipping.")
            continue

        loaded_families[family_name] = {}

        for model_name, model_obj in model_dict.items():
            if isinstance(model_obj, dict):
                loaded_families[family_name][model_name] = {}
                for subname, submodel in model_obj.items():
                    filename = f"{model_name}_{subname}.pth"
                    path = os.path.join(folder_path, filename)
                    if os.path.exists(path):
                        submodel.load_state_dict(torch.load(path, map_location='cpu'))
                        submodel.eval()
                        loaded_families[family_name][model_name][subname] = submodel
                        print(f"✅ Loaded {family_name}/{model_name}_{subname}")
                    else:
                        print(f"❌ Missing: {path}")
            else:
                filename = f"{model_name}.pth"
                path = os.path.join(folder_path, filename)
                if os.path.exists(path):
                    model_obj.load_state_dict(torch.load(path, map_location='cpu'))
                    model_obj.eval()
                    loaded_families[family_name][model_name] = model_obj
                    print(f"✅ Loaded {family_name}/{model_name}")
                else:
                    print(f"❌ Missing: {path}")

    return loaded_families

# classfier expr

In [ ]:
models={
    
    "PCAMLP": FeatureEncoder(
        extractor=PCAMLP(d_in=32*32, d_out=16, d_hidden=64, img_size=32, n_components=256),
        selector=TopKSelector(d_model=16, k=8)),
     "MiniViT": FeatureEncoder(
        extractor=MiniVisionTransformer(d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        selector=TopKSelector(d_model=16, k=8)),
    "MiniUNet": FeatureEncoder(
        extractor=MiniUNet(d_in=1, d_out=12, d_hidden=28,kernel_num=3, img_size=32),
        selector=TopKSelector(d_model=12, k=8)),
   
    
    "MLP": FeatureEncoder(
        extractor=MLPExtractor(d_in=32*32, d_out=10, d_hidden=17, img_size=32),
        selector=TopKSelector(d_model=10, k=8)),    
  
    
}
for key,values in models.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

full_dataset = datasets.MNIST(root="./data", train=True, download=False, transform=transform)

In [ ]:
runner = ExperimentRunner(
    models=models,
    dataset=full_dataset,
    num_folds=5,
    num_epochs=5,
    batch_size=64,
    lr=1e-3,
    classfier_in=8
)


In [ ]:
classifier = UniversalClassifier(d_in=8, d_out=10).to(device)

In [ ]:
runner.run()

# flow matching test

## expr without selector

In [ ]:
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

In [ ]:
models = {
    "MiniViT": VecField(
        matcher=Matcher(
            d_in=16,
            latent_model=MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32
            ),
            out_channels=1
        )
    ),
    "MiniUNet": VecField(
        matcher=Matcher(
            d_in=16,
            latent_model=MiniUNet(
                    d_in=1, d_out=16, d_hidden=27, kernel_num=3, img_size=32
                ),
            out_channels=1
        )
    ),
    "PCAMLP": VecField(
        matcher=Matcher(
            d_in=16,
            latent_model=PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=60, img_size=32, n_components=256
                ),
            out_channels=1
        )
    ),
    "MLP": VecField(
        matcher=Matcher(
            d_in=14,
            latent_model=MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=17, img_size=32
                ),
            out_channels=1
        )
    ),
}
for key,values in models.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

In [ ]:
runner = FlowExperimentRunner(
    models=models,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()

# flow expr

In [7]:
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

## expr-origin

In [8]:
modelo = {
    "MiniViT": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MiniUNet": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
                selector=TopKSelector(d_model=16, k=8)  #
            ),
            out_channels=1
        )
    ),
    "PCAMLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ),
                selector=TopKSelector(d_model=16, k=8)
            ),
            out_channels=1
        )
    ),
    "MLP": VecField(
        matcher=Matcher(
            d_in=8,
            latent_model=FeatureEncoder(
                extractor=MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
                selector=TopKSelector(d_model=14, k=8)
            ),
            out_channels=1
        )
    ),
}

In [9]:
for key,values in modelo.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

MiniViT model size: 0.10 MiB
MiniViT model params: 25941
MiniUNet model size: 0.10 MiB
MiniUNet model params: 26513
PCAMLP model size: 1.10 MiB
PCAMLP model params: 25694
MLP model size: 0.10 MiB
MLP model params: 26491


In [10]:
runner = FlowExperimentRunner(
    models=modelo,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1177.396: : 2500it [06:17,  6.63it/s, train=1177.3958, val=1174.0123]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1185.248: : 2500it [08:25,  4.95it/s, train=1185.2478, val=1191.4456]



>>> Training Model: PCAMLP


0it [00:00, ?it/s]/Users/aaaa/Downloads/modular_nn_experiment/models_.py:341: UserWarning: The operator 'aten::linalg_svd' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:14.)
  U, S, Vh = torch.linalg.svd(x_centered, full_matrices=False)
Epoch 2499, loss: 1197.371: : 2500it [03:10, 13.12it/s, train=1197.3713, val=1191.9229]



>>> Training Model: MLP


Epoch 2499, loss: 1217.305: : 2500it [03:39, 11.37it/s, train=1217.3051, val=1222.4648]


In [11]:
save_complex_models({"modelo": modelo }, save_dir="saved_models1")


Saving modelo / MiniViT to saved_models1/modelo/MiniViT.pth
Saving modelo / MiniUNet to saved_models1/modelo/MiniUNet.pth
Saving modelo / PCAMLP to saved_models1/modelo/PCAMLP.pth
Saving modelo / MLP to saved_models1/modelo/MLP.pth
✔️ Models saved to saved_models1


In [12]:
from torch.nn.functional import pairwise_distance,cosine_similarity

In [13]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t) 
ut_ref = path.conditional_vector_field(x,z,t) 

In [14]:
for key, value in modelo.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1170.5625
  MSE: 1.14312744140625
  Cosine Similarity: 0.6250811219215393
Model: MiniUNet
  Loss: 1187.80126953125
  MSE: 1.1599621772766113
  Cosine Similarity: 0.617703914642334
Model: PCAMLP
  Loss: 1185.7022705078125
  MSE: 1.1579123735427856
  Cosine Similarity: 0.6190270185470581
Model: MLP
  Loss: 1215.864013671875
  MSE: 1.1873672008514404
  Cosine Similarity: 0.6060708165168762


## expr-1 forward path end2end

In [15]:
models_full={ "MiniViT":
    vecfield( LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),Utdecoder(16,1)),
    "MiniUNet":
    vecfield( LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),Utdecoder(16,1)),
    "PCAMLP":
    vecfield(LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),Utdecoder(16,1)),
    "MLP":
    vecfield( LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),Utdecoder(14,1)),
    

   }

In [16]:
for key,values in models_full.items():
    print(f"{key} model size: {model_size_b(values) / MiB:.2f} MiB")
    print(f"{key} model params: {count_model_params(values)}")

MiniViT model size: 0.13 MiB
MiniViT model params: 33185
MiniUNet model size: 0.13 MiB
MiniUNet model params: 33757
PCAMLP model size: 1.13 MiB
PCAMLP model params: 32938
MLP model size: 0.12 MiB
MLP model params: 31905


In [17]:
runner = FlowExperimentRunner(
    models=models_full,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1140.636: : 2500it [05:19,  7.83it/s, train=1140.6362, val=1139.0303]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1175.140: : 2500it [05:59,  6.95it/s, train=1175.1404, val=1188.4213]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1167.458: : 2500it [03:03, 13.61it/s, train=1167.4576, val=1176.3792]



>>> Training Model: MLP


Epoch 2499, loss: 1178.629: : 2500it [03:26, 12.10it/s, train=1178.6290, val=1179.2749]


In [18]:
save_complex_models({"models_full": models_full }, save_dir="saved_models1")


Saving models_full / MiniViT to saved_models1/models_full/MiniViT.pth
Saving models_full / MiniUNet to saved_models1/models_full/MiniUNet.pth
Saving models_full / PCAMLP to saved_models1/models_full/PCAMLP.pth
Saving models_full / MLP to saved_models1/models_full/MLP.pth
✔️ Models saved to saved_models1


## expr-2 latent ->inverse

In [19]:
latentmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}

In [20]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Forward model size: {model_size_b(forward_model) / MiB:.2f} MiB")
    print(f"  Forward model params: {count_model_params(forward_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(forward_model) + count_model_params(inverse_model)
    total_size=(model_size_b(forward_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Forward model size: 0.06 MiB
  Forward model params: 16008
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 31832
  Total combined size: 0.13 MiB

[ConvNet]
  Forward model size: 0.06 MiB
  Forward model params: 16580
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 32976
  Total combined size: 0.13 MiB

[PCA_MLP]
  Forward model size: 1.06 MiB
  Forward model params: 15761
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 31611
  Total combined size: 2.13 MiB

[MLP]
  Forward model size: 0.06 MiB
  Forward model params: 16799
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 33437
  Total combined size: 0.13 MiB


In [21]:
runner = LatentFlowExperimentRunner(
    models=latentmodels,
    cfg_class=LatentCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 4999, loss: 0.201: : 5000it [06:46, 12.30it/s, train=0.2009, val=0.2058]     



>>> Training Model: ConvNet


Epoch 4999, loss: 0.093: : 5000it [05:45, 14.48it/s, train=0.0933, val=0.0950]     



>>> Training Model: PCA_MLP


Epoch 4999, loss: 0.001: : 5000it [02:14, 37.30it/s, train=0.0012, val=0.0010]



>>> Training Model: MLP


Epoch 4999, loss: 0.000: : 5000it [02:20, 35.54it/s, train=0.0001, val=0.0003]


In [22]:
save_complex_models({"latentmodels": latentmodels}, save_dir="saved_models1")

Saving latentmodels / MiniViT_forward to saved_models1/latentmodels/MiniViT_forward.pth
Saving latentmodels / MiniViT_inverse to saved_models1/latentmodels/MiniViT_inverse.pth
Saving latentmodels / ConvNet_forward to saved_models1/latentmodels/ConvNet_forward.pth
Saving latentmodels / ConvNet_inverse to saved_models1/latentmodels/ConvNet_inverse.pth
Saving latentmodels / PCA_MLP_forward to saved_models1/latentmodels/PCA_MLP_forward.pth
Saving latentmodels / PCA_MLP_inverse to saved_models1/latentmodels/PCA_MLP_inverse.pth
Saving latentmodels / MLP_forward to saved_models1/latentmodels/MLP_forward.pth
Saving latentmodels / MLP_inverse to saved_models1/latentmodels/MLP_inverse.pth
✔️ Models saved to saved_models1


In [23]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t) 
ut_ref = path.conditional_vector_field(x,z,t) 

In [24]:
from torch.nn.functional import pairwise_distance,cosine_similarity

In [25]:
for key, models_pair in latentmodels.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")
    

Model: MiniViT
  Loss: 1.1516457796096802
  MSE: 0.001107351970858872
  KL Divergence (forward): 0.0005423541297204792
  KL Divergence (inverse): 0.000543337722774595
  Wasserstein Distance: 0.9820559620857239
  JS Divergence: 0.00012784128193743527
  Cosine Similarity: -0.00021124798513483256
  Bhattacharyya Distance: 0.00013830333773512393
Model: ConvNet
  Loss: 0.053564105182886124
  MSE: 1.3077175935904961e-05
  KL Divergence (forward): -3.4453420084901154e-05
  KL Divergence (inverse): -3.4453943953849375e-05
  Wasserstein Distance: 0.21013964712619781
  JS Divergence: -3.934661435778253e-05
  Cosine Similarity: 0.01306759100407362
  Bhattacharyya Distance: 1.6343021798093105e-06
Model: PCA_MLP
  Loss: 0.0009749000892043114
  MSE: 6.093125921324827e-05
  KL Divergence (forward): 2.8457647204049863e-05
  KL Divergence (inverse): 2.838357067957986e-05
  Wasserstein Distance: 0.019721439108252525
  JS Divergence: 6.9945376708346885e-06
  Cosine Similarity: 0.0019299862906336784
  Bha

## expr-3 inverse-decoder

In [26]:
inversemodels = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": copy.deepcopy(latentmodels['MiniViT']['inverse'])
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": copy.deepcopy(latentmodels['ConvNet']['inverse'])
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": copy.deepcopy(latentmodels['PCA_MLP']['inverse'])
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": copy.deepcopy(latentmodels['MLP']['inverse'])
    }
}

In [27]:
for key, models_pair in inversemodels.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Decoder model size: {model_size_b(decoder_model) / MiB:.2f} MiB")
    print(f"  Decoder model params: {count_model_params(decoder_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(decoder_model) + count_model_params(inverse_model)
    total_size=(model_size_b(decoder_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


    
    


[MiniViT]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 33001
  Total combined size: 0.13 MiB

[ConvNet]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 33573
  Total combined size: 0.13 MiB

[PCA_MLP]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 33027
  Total combined size: 1.13 MiB

[MLP]
  Decoder model size: 0.06 MiB
  Decoder model params: 15106
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 31744
  Total combined size: 0.12 MiB


In [28]:
runner = InverseFlowExperimentRunner(
  models=inversemodels,
    cfg_class=InverseutCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3)
runner.run()


>>> Training Model: MiniViT


0it [00:00, ?it/s]

Epoch 4999, loss: 1308.280: : 5000it [07:24, 11.25it/s, train=1308.2795, val=1308.9442]



>>> Training Model: ConvNet


Epoch 4999, loss: 1364.348: : 5000it [06:37, 12.58it/s, train=1364.3481, val=1363.7729]



>>> Training Model: PCA_MLP


Epoch 4999, loss: 1341.609: : 5000it [03:12, 26.01it/s, train=1341.6091, val=1347.0945]



>>> Training Model: MLP


Epoch 4999, loss: 1268.920: : 5000it [03:11, 26.08it/s, train=1268.9199, val=1278.5266]  


In [29]:
save_complex_models({"inversemodels": inversemodels}, save_dir="saved_models1")


Saving inversemodels / MiniViT_decoder to saved_models1/inversemodels/MiniViT_decoder.pth
Saving inversemodels / MiniViT_inverse to saved_models1/inversemodels/MiniViT_inverse.pth
Saving inversemodels / ConvNet_decoder to saved_models1/inversemodels/ConvNet_decoder.pth
Saving inversemodels / ConvNet_inverse to saved_models1/inversemodels/ConvNet_inverse.pth
Saving inversemodels / PCA_MLP_decoder to saved_models1/inversemodels/PCA_MLP_decoder.pth
Saving inversemodels / PCA_MLP_inverse to saved_models1/inversemodels/PCA_MLP_inverse.pth
Saving inversemodels / MLP_decoder to saved_models1/inversemodels/MLP_decoder.pth
Saving inversemodels / MLP_inverse to saved_models1/inversemodels/MLP_inverse.pth
✔️ Models saved to saved_models1


In [30]:
for key, models_pair in inversemodels.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    ut_inverse = inverse_model(ut_ref)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    #loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    print(f"  Loss inverse-decoder: {loss2}")
   # print(f"  Loss forward-inverse: {loss3}")
    cos_sim = cosine_similarity(
        rearrange(ut_theta_i, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

  Loss inverse-decoder: 1300.522216796875
  Cosine Similarity: 0.5708125233650208
  Loss inverse-decoder: 1362.83984375
  Cosine Similarity: 0.5416808724403381
  Loss inverse-decoder: 1348.6861572265625
  Cosine Similarity: 0.5492115020751953
  Loss inverse-decoder: 1279.3509521484375
  Cosine Similarity: 0.5807086229324341


## expr-4 combnine encoder decoder 

In [31]:
mms = {
    "MiniViT":  vecfield(copy.deepcopy(latentmodels['MiniViT']['forward']),copy.deepcopy(inversemodels['MiniViT']['decoder'])),
    
    "MiniUNet":vecfield(copy.deepcopy(latentmodels['ConvNet']['forward']),copy.deepcopy(inversemodels['ConvNet']['decoder'])),
    "PCAMLP": vecfield(copy.deepcopy(latentmodels['PCA_MLP']['forward']),copy.deepcopy(inversemodels['PCA_MLP']['decoder'])),
    "MLP":vecfield(copy.deepcopy(latentmodels['MLP']['forward']),copy.deepcopy(inversemodels['MLP']['decoder'])),
}

In [32]:
for key, value in mms.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1507.434814453125
  MSE: 1.4721043109893799
  Cosine Similarity: 0.4883361756801605
Model: MiniUNet
  Loss: 1366.540283203125
  MSE: 1.3345119953155518
  Cosine Similarity: 0.5399709939956665
Model: PCAMLP
  Loss: 1490.6363525390625
  MSE: 1.4556996822357178
  Cosine Similarity: 0.5112177133560181
Model: MLP
  Loss: 5292.22802734375
  MSE: 5.168191432952881
  Cosine Similarity: 0.513841450214386


In [33]:
z, y = path.p_data.sample(1000) 
t = torch.rand(1000,1,1,1).to(z)
x = path.sample_conditional_path(z,t)
ut_ref = path.conditional_vector_field(x,z,t) 

In [34]:
runner = FlowExperimentRunner(
    models=mms,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1265.949: : 2500it [04:22,  9.52it/s, train=1265.9493, val=1254.6285]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1213.493: : 2500it [04:50,  8.59it/s, train=1213.4928, val=1217.5277]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1250.075: : 2500it [02:42, 15.37it/s, train=1250.0746, val=1264.2650]



>>> Training Model: MLP


Epoch 2499, loss: 1279.310: : 2500it [02:42, 15.41it/s, train=1279.3101, val=1290.2367]


In [35]:
save_complex_models({"mms": mms}, save_dir="saved_models1")


Saving mms / MiniViT to saved_models1/mms/MiniViT.pth
Saving mms / MiniUNet to saved_models1/mms/MiniUNet.pth
Saving mms / PCAMLP to saved_models1/mms/PCAMLP.pth
Saving mms / MLP to saved_models1/mms/MLP.pth
✔️ Models saved to saved_models1


In [36]:
for key, value in mms.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1254.934326171875
  MSE: 1.2255218029022217
  Cosine Similarity: 0.5908510088920593
Model: MiniUNet
  Loss: 1216.5355224609375
  MSE: 1.1880229711532593
  Cosine Similarity: 0.6072620749473572
Model: PCAMLP
  Loss: 1251.3232421875
  MSE: 1.2219953536987305
  Cosine Similarity: 0.5929058194160461
Model: MLP
  Loss: 1268.9385986328125
  MSE: 1.239198088645935
  Cosine Similarity: 0.5852565765380859


In [37]:
cms = {
    "MiniViT": {
        "forward":copy.deepcopy(latentmodels['MiniViT']['forward']) ,
        "inverse": copy.deepcopy(latentmodels['MiniViT']['inverse']),
        "decoder":copy.deepcopy(inversemodels['MiniViT']['decoder']),
    },
    "UNet": {
        "forward": copy.deepcopy(latentmodels['ConvNet']['forward']),
        "inverse":copy.deepcopy(latentmodels['ConvNet']['inverse']),
         "decoder":copy.deepcopy(inversemodels['ConvNet']['decoder']),
    },
    "PCA_MLP": {
         "forward": copy.deepcopy(latentmodels['PCA_MLP']['forward']),
        "inverse":copy.deepcopy(latentmodels['PCA_MLP']['inverse']),
         "decoder":copy.deepcopy(inversemodels['PCA_MLP']['decoder']),
    },
    "MLP": {
        "forward": copy.deepcopy(latentmodels['MLP']['forward']),
        "inverse":copy.deepcopy(latentmodels['MLP']['inverse']),
         "decoder":copy.deepcopy(inversemodels['MLP']['decoder']),
    }
}

In [38]:
for key, value in cms.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1507.069580078125
  Loss inverse-decoder: 1301.132080078125
  Loss forward-inverse: 208.46871948242188
  MSE: 1.471747636795044
  Cosine Similarity: 0.48802468180656433
Model: UNet
  Loss forward-decoder: 1366.967041015625
  Loss inverse-decoder: 1362.8819580078125
  Loss forward-inverse: 3.873908519744873
  MSE: 1.3349287509918213
  Cosine Similarity: 0.5394882559776306
Model: PCA_MLP
  Loss forward-decoder: 1466.791259765625
  Loss inverse-decoder: 1349.8631591796875
  Loss forward-inverse: 116.73384857177734
  MSE: 1.4324133396148682
  Cosine Similarity: 0.5124450922012329
Model: MLP
  Loss forward-decoder: 5996.01123046875
  Loss inverse-decoder: 1280.009033203125
  Loss forward-inverse: 4676.42822265625
  MSE: 5.855479717254639
  Cosine Similarity: 0.5150271058082581


In [39]:
runner = FullFlowExperimentRunner(
    models=cms,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=2500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 537.068: : 2500it [06:28,  6.44it/s, train=537.0682, val=519.1955]     



>>> Training Model: UNet


Epoch 2499, loss: 416.735: : 2500it [04:29,  9.26it/s, train=416.7350, val=420.2726]



>>> Training Model: PCA_MLP


Epoch 2499, loss: 452.975: : 2500it [02:20, 17.79it/s, train=452.9746, val=457.1654]  



>>> Training Model: MLP


Epoch 2499, loss: 525.409: : 2500it [02:18, 18.00it/s, train=525.4085, val=678.4218]   


In [40]:
save_complex_models({"cms": cms}, save_dir="saved_models1")


Saving cms / MiniViT_forward to saved_models1/cms/MiniViT_forward.pth
Saving cms / MiniViT_inverse to saved_models1/cms/MiniViT_inverse.pth
Saving cms / MiniViT_decoder to saved_models1/cms/MiniViT_decoder.pth
Saving cms / UNet_forward to saved_models1/cms/UNet_forward.pth
Saving cms / UNet_inverse to saved_models1/cms/UNet_inverse.pth
Saving cms / UNet_decoder to saved_models1/cms/UNet_decoder.pth
Saving cms / PCA_MLP_forward to saved_models1/cms/PCA_MLP_forward.pth
Saving cms / PCA_MLP_inverse to saved_models1/cms/PCA_MLP_inverse.pth
Saving cms / PCA_MLP_decoder to saved_models1/cms/PCA_MLP_decoder.pth
Saving cms / MLP_forward to saved_models1/cms/MLP_forward.pth
Saving cms / MLP_inverse to saved_models1/cms/MLP_inverse.pth
Saving cms / MLP_decoder to saved_models1/cms/MLP_decoder.pth
✔️ Models saved to saved_models1


In [41]:
for key, value in cms.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1427.3499755859375
  Loss inverse-decoder: 1316.1529541015625
  Loss forward-inverse: 167.1667022705078
  MSE: 1.3938965797424316
  Cosine Similarity: 0.5198264122009277
Model: UNet
  Loss forward-decoder: 1252.224609375
  Loss inverse-decoder: 1223.9212646484375
  Loss forward-inverse: 16.008804321289062
  MSE: 1.2228755950927734
  Cosine Similarity: 0.591926097869873
Model: PCA_MLP
  Loss forward-decoder: 1358.70556640625
  Loss inverse-decoder: 1271.799560546875
  Loss forward-inverse: 96.99628448486328
  MSE: 1.3268609046936035
  Cosine Similarity: 0.5612391829490662
Model: MLP
  Loss forward-decoder: 1333.4827880859375
  Loss inverse-decoder: 1276.3409423828125
  Loss forward-inverse: 59.907386779785156
  MSE: 1.3022292852401733
  Cosine Similarity: 0.5685890316963196


## for-inv-full

In [42]:
fullmodels = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),
        "decoder":Utdecoder(16,1)
    },
    "UNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),
         "decoder":Utdecoder(16,1)
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256
                ),
         "decoder":Utdecoder(16,1)
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ),
         "decoder":Utdecoder(14,1) 
    }
}

In [43]:
runner = FullFlowExperimentRunner(
    models=fullmodels,
    cfg_class=FullCFG,
    path=path,           
    num_epochs=2500,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 630.663: : 2500it [06:22,  6.53it/s, train=630.6628, val=633.7054]



>>> Training Model: UNet


Epoch 2499, loss: 636.051: : 2500it [04:30,  9.25it/s, train=636.0510, val=627.0675]



>>> Training Model: PCA_MLP


Epoch 2499, loss: 550.252: : 2500it [02:20, 17.85it/s, train=550.2522, val=549.5233]



>>> Training Model: MLP


Epoch 2499, loss: 486.541: : 2500it [02:17, 18.15it/s, train=486.5405, val=490.4475]


In [44]:
save_complex_models({"fullmodels": fullmodels}, save_dir="saved_models1")


Saving fullmodels / MiniViT_forward to saved_models1/fullmodels/MiniViT_forward.pth
Saving fullmodels / MiniViT_inverse to saved_models1/fullmodels/MiniViT_inverse.pth
Saving fullmodels / MiniViT_decoder to saved_models1/fullmodels/MiniViT_decoder.pth
Saving fullmodels / UNet_forward to saved_models1/fullmodels/UNet_forward.pth
Saving fullmodels / UNet_inverse to saved_models1/fullmodels/UNet_inverse.pth
Saving fullmodels / UNet_decoder to saved_models1/fullmodels/UNet_decoder.pth
Saving fullmodels / PCA_MLP_forward to saved_models1/fullmodels/PCA_MLP_forward.pth
Saving fullmodels / PCA_MLP_inverse to saved_models1/fullmodels/PCA_MLP_inverse.pth
Saving fullmodels / PCA_MLP_decoder to saved_models1/fullmodels/PCA_MLP_decoder.pth
Saving fullmodels / MLP_forward to saved_models1/fullmodels/MLP_forward.pth
Saving fullmodels / MLP_inverse to saved_models1/fullmodels/MLP_inverse.pth
Saving fullmodels / MLP_decoder to saved_models1/fullmodels/MLP_decoder.pth
✔️ Models saved to saved_models1


In [45]:
ams= {
    "MiniViT":  vecfield(copy.deepcopy(fullmodels['MiniViT']['forward']),copy.deepcopy(fullmodels['MiniViT']['decoder'])),
    
    "MiniUNet":vecfield(copy.deepcopy(fullmodels['UNet']['forward']),copy.deepcopy(fullmodels['UNet']['decoder'])),
    "PCAMLP": vecfield(copy.deepcopy(fullmodels['PCA_MLP']['forward']),copy.deepcopy(fullmodels['PCA_MLP']['decoder'])),
    "MLP":vecfield(copy.deepcopy(fullmodels['MLP']['forward']),copy.deepcopy(fullmodels['MLP']['decoder'])),
}

In [46]:
for key, value in fullmodels.items():
    forward_model = value["forward"]
    inverse_model = value["inverse"]
    decoder_model = value["decoder"]
    
    # Forward and inverse outputs
    ut_forward = forward_model(x, t=t, y=y)
    ut_inverse = inverse_model(ut_ref)
    ut_theta = decoder_model(ut_forward,t=t,y=y)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b c h w -> b').mean()
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    mse = F.mse_loss(ut_theta, ut_ref)
    print(f"  Loss forward-decoder: {loss}")
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    print(f"  MSE: {mse}")
    
    cos_sim = cosine_similarity(
        rearrange(ut_theta, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss forward-decoder: 1906.2642822265625
  Loss inverse-decoder: 1883.153564453125
  Loss forward-inverse: 0.29644840955734253
  MSE: 1.8615862131118774
  Cosine Similarity: 0.4409172236919403
Model: UNet
  Loss forward-decoder: 1884.2164306640625
  Loss inverse-decoder: 1880.036865234375
  Loss forward-inverse: 0.022274235263466835
  MSE: 1.8400551080703735
  Cosine Similarity: 0.2231205701828003
Model: PCA_MLP
  Loss forward-decoder: 1604.7022705078125
  Loss inverse-decoder: 1578.2940673828125
  Loss forward-inverse: 5.7809319496154785
  MSE: 1.5670920610427856
  Cosine Similarity: 0.42934781312942505
Model: MLP
  Loss forward-decoder: 1427.4068603515625
  Loss inverse-decoder: 1414.9119873046875
  Loss forward-inverse: 8.634706497192383
  MSE: 1.3939520120620728
  Cosine Similarity: 0.5100666880607605


In [47]:
for key, value in ams.items():
    
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1906.2642822265625
  MSE: 1.8615862131118774
  Cosine Similarity: 0.4409172236919403
Model: MiniUNet
  Loss: 1884.2164306640625
  MSE: 1.8400551080703735
  Cosine Similarity: 0.2231205701828003
Model: PCAMLP
  Loss: 1604.7022705078125
  MSE: 1.5670920610427856
  Cosine Similarity: 0.42934781312942505
Model: MLP
  Loss: 1427.4068603515625
  MSE: 1.3939520120620728
  Cosine Similarity: 0.5100666880607605


## inverse path train

In [48]:
inversemodels2 = {
    "MiniViT": {
        "decoder": Utdecoder(16,1),
        "inverse": MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32)
    },
    "ConvNet": {
        "decoder":  Utdecoder(16,1),
        "inverse": MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32)
    },
    "PCA_MLP": {
        "decoder": Utdecoder(16,1),
        "inverse": PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=58, img_size=32, n_components=256)
    },
    "MLP": {
        "decoder":Utdecoder(14,1),
        "inverse": MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                )
    }
}


In [49]:
for key, models_pair in inversemodels2.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Decoder model size: {model_size_b(decoder_model) / MiB:.2f} MiB")
    print(f"  Decoder model params: {count_model_params(decoder_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(decoder_model) + count_model_params(inverse_model)
    total_size=(model_size_b(decoder_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 33001
  Total combined size: 0.13 MiB

[ConvNet]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 33573
  Total combined size: 0.13 MiB

[PCA_MLP]
  Decoder model size: 0.07 MiB
  Decoder model params: 17177
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 33027
  Total combined size: 1.13 MiB

[MLP]
  Decoder model size: 0.06 MiB
  Decoder model params: 15106
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 31744
  Total combined size: 0.12 MiB


In [50]:

runner = InverseFlowExperimentRunner2(
  models=inversemodels2,
    cfg_class=InverseutCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3)
runner.run()


>>> Training Model: MiniViT


0it [00:00, ?it/s]

Epoch 4999, loss: 1155.432: : 5000it [05:36, 14.85it/s, train=1155.4316, val=1150.2428]



>>> Training Model: ConvNet


Epoch 4999, loss: 1198.386: : 5000it [05:11, 16.07it/s, train=1198.3864, val=1182.5176]



>>> Training Model: PCA_MLP


Epoch 4999, loss: 1203.673: : 5000it [03:12, 26.01it/s, train=1203.6729, val=1200.9277]



>>> Training Model: MLP


Epoch 4999, loss: 1245.431: : 5000it [03:11, 26.17it/s, train=1245.4308, val=1239.3654]


In [51]:
save_complex_models({"inversemodels2": inversemodels2}, save_dir="saved_models1")


Saving inversemodels2 / MiniViT_decoder to saved_models1/inversemodels2/MiniViT_decoder.pth
Saving inversemodels2 / MiniViT_inverse to saved_models1/inversemodels2/MiniViT_inverse.pth
Saving inversemodels2 / ConvNet_decoder to saved_models1/inversemodels2/ConvNet_decoder.pth
Saving inversemodels2 / ConvNet_inverse to saved_models1/inversemodels2/ConvNet_inverse.pth
Saving inversemodels2 / PCA_MLP_decoder to saved_models1/inversemodels2/PCA_MLP_decoder.pth
Saving inversemodels2 / PCA_MLP_inverse to saved_models1/inversemodels2/PCA_MLP_inverse.pth
Saving inversemodels2 / MLP_decoder to saved_models1/inversemodels2/MLP_decoder.pth
Saving inversemodels2 / MLP_inverse to saved_models1/inversemodels2/MLP_inverse.pth
✔️ Models saved to saved_models1


In [71]:
for key, models_pair in inversemodels2.items():
    decoder_model = models_pair["decoder"]
    inverse_model = models_pair["inverse"]
    ut_inverse = inverse_model(ut_ref)
    ut_theta_i=decoder_model(ut_inverse,t=t,y=y)
    loss2=einsum(torch.square(  ut_theta_i - ut_ref), 'b c h w -> b').mean()
    loss3=einsum(torch.square(  ut_theta_i - ut_theta), 'b c h w -> b').mean()
    print(f"  Loss inverse-decoder: {loss2}")
    print(f"  Loss forward-inverse: {loss3}")
    cos_sim = cosine_similarity(
        rearrange(ut_theta_i, 'b c h w -> b (c h w)'),
        rearrange(ut_ref, 'b c h w -> b (c h w)')
    )
    print(f"  Cosine Similarity: {cos_sim.mean()}")

  Loss inverse-decoder: 1148.623046875
  Loss forward-inverse: 132.15028381347656
  Cosine Similarity: 0.6354509592056274
  Loss inverse-decoder: 1194.87890625
  Loss forward-inverse: 126.50586700439453
  Cosine Similarity: 0.6162397265434265
  Loss inverse-decoder: 1254.3046875
  Loss forward-inverse: 124.77191162109375
  Cosine Similarity: 0.5935111045837402
  Loss inverse-decoder: 1248.29248046875
  Loss forward-inverse: 94.8016128540039
  Cosine Similarity: 0.5939139723777771


In [70]:
save_complex_models({"inversemodels2": inversemodels2}, save_dir="saved_models1")

Saving inversemodels2 / MiniViT_decoder to saved_models1/inversemodels2/MiniViT_decoder.pth
Saving inversemodels2 / MiniViT_inverse to saved_models1/inversemodels2/MiniViT_inverse.pth
Saving inversemodels2 / ConvNet_decoder to saved_models1/inversemodels2/ConvNet_decoder.pth
Saving inversemodels2 / ConvNet_inverse to saved_models1/inversemodels2/ConvNet_inverse.pth
Saving inversemodels2 / PCA_MLP_decoder to saved_models1/inversemodels2/PCA_MLP_decoder.pth
Saving inversemodels2 / PCA_MLP_inverse to saved_models1/inversemodels2/PCA_MLP_inverse.pth
Saving inversemodels2 / MLP_decoder to saved_models1/inversemodels2/MLP_decoder.pth
Saving inversemodels2 / MLP_inverse to saved_models1/inversemodels2/MLP_inverse.pth
✔️ Models saved to saved_models1


In [53]:
latentmodels2 = {
    "MiniViT": {
        "forward": LatentVecField(MiniVisionTransformer(
                    d_in=1, d_out=16, d_hidden=32, patch_size=4, num_heads=4, img_size=32),16),
        "inverse": copy.deepcopy(inversemodels2["MiniViT"]['inverse'])
    },
    "ConvNet": {
        "forward": LatentVecField(MiniUNet(
                    d_in=1, d_out=16, d_hidden=26, kernel_num=3, img_size=32
                ),16),
        "inverse": copy.deepcopy(inversemodels2["ConvNet"]['inverse'])
                ,
    },
    "PCA_MLP": {
        "forward": LatentVecField(PCAMLP(
                    d_in=32*32, d_out=16, d_hidden=57, img_size=32, n_components=256
                ), 16),
        "inverse":copy.deepcopy(inversemodels2["PCA_MLP"]['inverse'])
    },
    "MLP": {
        "forward": LatentVecField(MLPExtractor(
                    d_in=32*32, d_out=14, d_hidden=16, img_size=32
                ), 14),
        "inverse":copy.deepcopy(inversemodels2["MLP"]['inverse'])
    }
}


In [54]:
for key, models_pair in latentmodels2.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    
    print(f"\n[{key}]")
    
    print(f"  Forward model size: {model_size_b(forward_model) / MiB:.2f} MiB")
    print(f"  Forward model params: {count_model_params(forward_model)}")
    
    print(f"  Inverse model size: {model_size_b(inverse_model) / MiB:.2f} MiB")
    print(f"  Inverse model params: {count_model_params(inverse_model)}")
    
    total_params = count_model_params(forward_model) + count_model_params(inverse_model)
    total_size=(model_size_b(forward_model)+model_size_b(inverse_model))/ MiB
    print(f"  Total combined params: {total_params}")
    print(f"  Total combined size: {total_size:.2f} MiB")


[MiniViT]
  Forward model size: 0.06 MiB
  Forward model params: 16008
  Inverse model size: 0.06 MiB
  Inverse model params: 15824
  Total combined params: 31832
  Total combined size: 0.13 MiB

[ConvNet]
  Forward model size: 0.06 MiB
  Forward model params: 16580
  Inverse model size: 0.06 MiB
  Inverse model params: 16396
  Total combined params: 32976
  Total combined size: 0.13 MiB

[PCA_MLP]
  Forward model size: 1.06 MiB
  Forward model params: 15761
  Inverse model size: 1.06 MiB
  Inverse model params: 15850
  Total combined params: 31611
  Total combined size: 2.13 MiB

[MLP]
  Forward model size: 0.06 MiB
  Forward model params: 16799
  Inverse model size: 0.06 MiB
  Inverse model params: 16638
  Total combined params: 33437
  Total combined size: 0.13 MiB


In [75]:
runner = LatentFlowExperimentRunner2(
    models=latentmodels2,
    cfg_class=LatentCFG,
    path=path,           
    num_epochs=5000,
    batch_size=128,
    device=device,
    eta=0.1,
    lr=1e-3
)

runner.run()


>>> Training Model: MiniViT


Epoch 4999, loss: 472.667: : 5000it [05:25, 15.36it/s, train=472.6667, val=464.5105]



>>> Training Model: ConvNet


Epoch 4999, loss: 423.848: : 5000it [05:33, 15.01it/s, train=423.8480, val=420.3210]



>>> Training Model: PCA_MLP


Epoch 4999, loss: 11.375: : 5000it [02:30, 33.25it/s, train=11.3749, val=10.8729]



>>> Training Model: MLP


Epoch 4999, loss: 6.115: : 5000it [02:18, 36.07it/s, train=6.1147, val=5.6416]


In [86]:
save_complex_models({"latentmodels2": latentmodels2}, save_dir="saved_models1")


Saving latentmodels2 / MiniViT_forward to saved_models1/latentmodels2/MiniViT_forward.pth
Saving latentmodels2 / MiniViT_inverse to saved_models1/latentmodels2/MiniViT_inverse.pth
Saving latentmodels2 / ConvNet_forward to saved_models1/latentmodels2/ConvNet_forward.pth
Saving latentmodels2 / ConvNet_inverse to saved_models1/latentmodels2/ConvNet_inverse.pth
Saving latentmodels2 / PCA_MLP_forward to saved_models1/latentmodels2/PCA_MLP_forward.pth
Saving latentmodels2 / PCA_MLP_inverse to saved_models1/latentmodels2/PCA_MLP_inverse.pth
Saving latentmodels2 / MLP_forward to saved_models1/latentmodels2/MLP_forward.pth
Saving latentmodels2 / MLP_inverse to saved_models1/latentmodels2/MLP_inverse.pth
✔️ Models saved to saved_models1


In [87]:
for key, models_pair in latentmodels2.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]
    forward_model.eval()
    inverse_model.eval()
    

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

Model: MiniViT
  Loss: 1987.2706298828125
  MSE: 1.9108370542526245
Model: ConvNet
  Loss: 2165.460205078125
  MSE: 0.5286768674850464
Model: PCA_MLP
  Loss: 42.38883972167969
  MSE: 2.6493024826049805
Model: MLP
  Loss: 15.6226224899292
  MSE: 1.1159015893936157


In [83]:
for key, models_pair in latentmodels2.items():
    forward_model = models_pair["forward"]
    inverse_model = models_pair["inverse"]

    # Forward and inverse outputs
    ut_forward = forward_model(x, t, y)
    ut_inverse = inverse_model(x)

    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_forward - ut_inverse), 'b seq d -> b').mean()
    mse = F.mse_loss(ut_forward, ut_inverse)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")

    # Reshape for probability-based metrics
    ut_forward = rearrange(ut_forward, 'b seq d -> b (seq d)')
    ut_inverse = rearrange(ut_inverse, 'b seq d -> b (seq d)')
    ut_forward_prob = torch.softmax(ut_forward, dim=-1)
    ut_inverse_prob = torch.softmax(ut_inverse, dim=-1)

    # KL Divergence
    kl_div_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (ut_inverse_prob + 1e-8)), dim=-1)
    kl_div_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (ut_forward_prob + 1e-8)), dim=-1)
    print(f"  KL Divergence (forward): {kl_div_forward.mean()}")
    print(f"  KL Divergence (inverse): {kl_div_inverse.mean()}")

    # Wasserstein Distance
    wasserstein_distance = pairwise_distance(ut_forward, ut_inverse, p=2).mean()
    print(f"  Wasserstein Distance: {wasserstein_distance}")

    # JS Divergence
    m = 0.5 * (ut_forward_prob + ut_inverse_prob)
    kl_forward = torch.sum(ut_forward_prob * torch.log(ut_forward_prob / (m + 1e-8)), dim=-1)
    kl_inverse = torch.sum(ut_inverse_prob * torch.log(ut_inverse_prob / (m + 1e-8)), dim=-1)
    js_divergence = 0.5 * (kl_forward + kl_inverse)
    print(f"  JS Divergence: {js_divergence.mean()}")

    # Cosine Similarity
    cos_sim = cosine_similarity(ut_forward, ut_inverse, dim=-1)
    print(f"  Cosine Similarity: {cos_sim.mean()}")

    # Bhattacharyya Distance
    bc_coefficient = torch.sum(torch.sqrt(ut_forward_prob * ut_inverse_prob), dim=-1)
    bhattacharyya_distance = -torch.log(bc_coefficient + 1e-8)
    print(f"  Bhattacharyya Distance: {bhattacharyya_distance.mean()}")

Model: MiniViT
  Loss: 1987.2706298828125
  MSE: 1.9108370542526245
  KL Divergence (forward): 1.2715821266174316
  KL Divergence (inverse): 1.118300199508667
  Wasserstein Distance: 42.515625
  JS Divergence: 0.21550272405147552
  Cosine Similarity: 0.6048224568367004
  Bhattacharyya Distance: 0.294979065656662
Model: ConvNet
  Loss: 2165.460205078125
  MSE: 0.5286768674850464
  KL Divergence (forward): 0.23340752720832825
  KL Divergence (inverse): 0.23926831781864166
  Wasserstein Distance: 45.282859802246094
  JS Divergence: 0.054949503391981125
  Cosine Similarity: 0.16279414296150208
  Bhattacharyya Distance: 0.05882152169942856
Model: PCA_MLP
  Loss: 42.38883972167969
  MSE: 2.6493024826049805
  KL Divergence (forward): 1.0180891752243042
  KL Divergence (inverse): 1.2604018449783325
  Wasserstein Distance: 6.321715354919434
  JS Divergence: 0.22446630895137787
  Cosine Similarity: 0.4811727702617645
  Bhattacharyya Distance: 0.30028337240219116
Model: MLP
  Loss: 15.62262248992

In [78]:
mms2 = {
    "MiniViT":  vecfield(copy.deepcopy(latentmodels2['MiniViT']['forward']),copy.deepcopy(inversemodels2['MiniViT']['decoder'])),
    
    "MiniUNet":vecfield(copy.deepcopy(latentmodels2['ConvNet']['forward']),copy.deepcopy(inversemodels2['ConvNet']['decoder'])),
    "PCAMLP": vecfield(latentmodels2['PCA_MLP']['forward'],inversemodels2['PCA_MLP']['decoder']),
    "MLP":vecfield(copy.deepcopy(latentmodels2['MLP']['forward']),copy.deepcopy(inversemodels2['MLP']['decoder'])),
}

In [79]:
for key, value in mms2.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1235.2884521484375
  MSE: 1.2063363790512085
  Cosine Similarity: 0.5993676781654358
Model: MiniUNet
  Loss: 1234.9598388671875
  MSE: 1.206015706062317
  Cosine Similarity: 0.599275529384613
Model: PCAMLP
  Loss: 1241.3961181640625
  MSE: 1.2123008966445923
  Cosine Similarity: 0.5968700051307678
Model: MLP
  Loss: 1254.8636474609375
  MSE: 1.2254527807235718
  Cosine Similarity: 0.5910233855247498


In [80]:
runner = FlowExperimentRunner(
    models=mms2,
    cfg_class=CFG,       
    path=path,
    eta=0.1,
    num_epochs=2500,
    batch_size=250,
    lr=1e-3,
    device=device
    
)
runner.run()


>>> Training Model: MiniViT


Epoch 2499, loss: 1131.409: : 2500it [04:38,  8.97it/s, train=1131.4088, val=1131.2349]



>>> Training Model: MiniUNet


Epoch 2499, loss: 1134.244: : 2500it [05:06,  8.15it/s, train=1134.2440, val=1146.6731]



>>> Training Model: PCAMLP


Epoch 2499, loss: 1164.752: : 2500it [03:13, 12.95it/s, train=1164.7521, val=1163.1658]



>>> Training Model: MLP


Epoch 2499, loss: 1160.465: : 2500it [02:54, 14.35it/s, train=1160.4653, val=1166.0339]


In [81]:
save_complex_models({"mms2": mms2}, save_dir="saved_models1")

Saving mms2 / MiniViT to saved_models1/mms2/MiniViT.pth
Saving mms2 / MiniUNet to saved_models1/mms2/MiniUNet.pth
Saving mms2 / PCAMLP to saved_models1/mms2/PCAMLP.pth
Saving mms2 / MLP to saved_models1/mms2/MLP.pth
✔️ Models saved to saved_models1


In [82]:
for key, value in mms2.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

Model: MiniViT
  Loss: 1131.5089111328125
  MSE: 1.1049892902374268
  Cosine Similarity: 0.6423801779747009
Model: MiniUNet
  Loss: 1136.9317626953125
  MSE: 1.1102849245071411
  Cosine Similarity: 0.6401692032814026
Model: PCAMLP
  Loss: 1157.2010498046875
  MSE: 1.1300792694091797
  Cosine Similarity: 0.6320523619651794
Model: MLP
  Loss: 1163.583740234375
  MSE: 1.1363122463226318
  Cosine Similarity: 0.6294465661048889


# save

In [52]:
import os

In [53]:
def save_complex_models(nested_dict, save_dir="saved_models"):
    os.makedirs(save_dir, exist_ok=True)

    for top_key, sub_dict in nested_dict.items():
        model_folder = os.path.join(save_dir, top_key)
        os.makedirs(model_folder, exist_ok=True)

        if isinstance(sub_dict, dict):
            for name, model in sub_dict.items():
               
                if isinstance(model, dict):
                    for subname, submodel in model.items():
                        path = os.path.join(model_folder, f"{name}_{subname}.pth")
                        print(f"Saving {top_key} / {name}_{subname} to {path}")
                        torch.save(submodel.state_dict(), path)
                else:
                    path = os.path.join(model_folder, f"{name}.pth")
                    print(f"Saving {top_key} / {name} to {path}")
                    torch.save(model.state_dict(), path)
        else:
          
            path = os.path.join(save_dir, f"{top_key}.pth")
            print(f"Saving {top_key} to {path}")
            torch.save(sub_dict.state_dict(), path)

    print(f"✔️ Models saved to {save_dir}")

In [54]:
models_group = {
    "modelo": modelo,
    "models_full": models_full,
    "latentmodels": latentmodels,
    "inversemodels": inversemodels,
    "mms": mms,
    "cms": cms,
    "inversemodels2":inversemodels2,
    "latentmodels2":latentmodels2,
      "mms2": mms2,
    
    
}



In [55]:
save_complex_models(models_group, save_dir="saved_models1")

Saving modelo / MiniViT to saved_models1/modelo/MiniViT.pth
Saving modelo / MiniUNet to saved_models1/modelo/MiniUNet.pth
Saving modelo / PCAMLP to saved_models1/modelo/PCAMLP.pth
Saving modelo / MLP to saved_models1/modelo/MLP.pth
Saving models_full / MiniViT to saved_models1/models_full/MiniViT.pth
Saving models_full / MiniUNet to saved_models1/models_full/MiniUNet.pth
Saving models_full / PCAMLP to saved_models1/models_full/PCAMLP.pth
Saving models_full / MLP to saved_models1/models_full/MLP.pth
Saving latentmodels / MiniViT_forward to saved_models1/latentmodels/MiniViT_forward.pth
Saving latentmodels / MiniViT_inverse to saved_models1/latentmodels/MiniViT_inverse.pth
Saving latentmodels / ConvNet_forward to saved_models1/latentmodels/ConvNet_forward.pth
Saving latentmodels / ConvNet_inverse to saved_models1/latentmodels/ConvNet_inverse.pth
Saving latentmodels / PCA_MLP_forward to saved_models1/latentmodels/PCA_MLP_forward.pth
Saving latentmodels / PCA_MLP_inverse to saved_models1/l

# expr

In [ ]:
z, y = path.p_data.sample(2000) 
t = torch.rand(2000,1,1,1).to(z)
x = path.sample_conditional_path(z,t) 
ut_ref = path.conditional_vector_field(x,z,t) 

In [ ]:
for key, value in modelo.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

In [ ]:
for key, value in models_full.items():
    
    # Forward and inverse outputs
    ut_theta = value(x, t=t, y=y)
    print(f"Model: {key}")
    
    # Loss and MSE
    loss = einsum(torch.square(ut_theta - ut_ref), 'b  c h w -> b').mean()
    mse = F.mse_loss(ut_theta ,ut_ref)
    print(f"  Loss: {loss}")
    print(f"  MSE: {mse}")
    cos_sim = cosine_similarity(rearrange(ut_theta, 'b c h w-> b (c h w)'), rearrange(ut_ref, 'b c h w -> b (c h w)'))
    print(f"  Cosine Similarity: {cos_sim.mean()}")

In [ ]:
 latentmodels['MiniViT']['inverse']

In [ ]:
print(latentmodels['MiniViT']['inverse']  is inversemodels['MiniViT']['inverse'])